# Установка

In [3]:
!pip install deeppavlov==1.7.0
!pip install transformers==4.30.0 huggingface_hub==0.36.2 tokenizers==0.13.2
!pip install protobuf==3.20.0 pytorch-crf==0.7.2 sentencepiece==0.2.0
!pip install nltk scikit-learn

# Импорты и данные

In [4]:
import pandas as pd
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

nltk.download('wordnet')
nltk.download('omw-1.4')

df = pd.read_csv('toxic_comments.csv', 
                 engine='python',
                 on_bad_lines='skip')

df = df[['text', 'toxic']].dropna()
df['toxic'] = pd.to_numeric(df['toxic'], errors='coerce').dropna().astype(int)
df = df.dropna()

print(df.shape)
print(df['toxic'].value_counts())
df.head()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Alex\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Alex\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


(113081, 2)
0    101586
1     11495
Name: toxic, dtype: int64


,text,toxic
0,Explanation\nWhy the edits made under my usern...,0
1,D'aww! He matches this background colour I'm s...,0
2,"Hey man, I'm really not trying to edit war. It...",0
3,"""\nMore\nI can't make any real suggestions on ...",0
4,"You, sir, are my hero. Any chance you remember...",0


# Лемматизация

In [5]:
import nltk
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = str(text).lower().split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t.isalpha()]
    return " ".join(tokens)

df['clean_text'] = df['text'].apply(preprocess)

print("До:  ", df['text'][0][:120])
print("После:", df['clean_text'][0][:120])

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Alex\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Alex\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


До:   Explanation
Why the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just clo
После: explanation why the edits made under my username hardcore metallica fan were they just closure on some gas after i voted


# Загрузка модели

In [6]:
from deeppavlov import build_model, configs

model = build_model(configs.classifiers.insults_kaggle_bert, download=True)
print("Модель загружена")

2026-04-27 18:59:34.564 INFO in 'deeppavlov.download'['download'] at line 138: Skipped http://files.deeppavlov.ai/deeppavlov_data/classifiers/insults_kaggle_torch_bert_v5.tar.gz download because of matching hashes
2026-04-27 18:59:45.687 INFO in 'deeppavlov.download'['download'] at line 138: Skipped http://files.deeppavlov.ai/datasets/insults_data.tar.gz download because of matching hashes
e:\labs\kafedra\.venv\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\labs\kafedra\.venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-uncased were not used when in

Модель загружена


# Проверка на примерах из датасета

In [10]:
samples = df.sample(10, random_state=42)

texts  = samples['clean_text'].tolist()
labels = samples['toxic'].tolist()

preds = model(texts)

for text, true, pred in zip(texts, labels, preds):
    predicted_toxic = 1 if pred == 'Insult' else 0
    match = "✓" if predicted_toxic == true else "✗"
    print(f"{match} [токсик?={'да' if true else ' нет':>5}] [модель={pred}]")
    print(f"  {text[:80]}")
    print()

✓ [токсик?=  нет] [модель=Not Insult]
  do what you but never get rid of a give your sister a kiss for

✓ [токсик?=  нет] [модель=Not Insult]
  moved discussion on origin the following wa inserted into the article by i do no

✓ [токсик?=  нет] [модель=Not Insult]
  other source i am listing source i have found some have information that can be 

✓ [токсик?=  нет] [модель=Not Insult]
  i understand what i i do not see i see try this in any culture switch to you wil

✗ [токсик?=   да] [модель=Not Insult]
  hitler is an

✓ [токсик?=  нет] [модель=Not Insult]
  considering that who is wa blocked for sock puppetry six month blocking

✓ [токсик?=  нет] [модель=Not Insult]
  of course you may

✓ [токсик?=  нет] [модель=Not Insult]
  barnstar the working barnstar i award this barnstar to two fantastic lady editor

✗ [токсик?=   да] [модель=Not Insult]
  if i saw ya spit on pretty much say it all

✓ [токсик?=  нет] [модель=Not Insult]
  check but van susteren got her award before she joined and

# Модель для русского текста

# Установка доп. библиотек

In [11]:
%pip install pymorphy2

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.2 MB 1.6 MB/s eta 0:00:05
   ------- -------------------------------- 1.6/8.2 MB 2.5 MB/s eta 0:00:03
   ------------ --------------------------- 2.6/8.2 MB 3.4 MB/s eta 0:00:02
   ---------------- ----------------------- 3.4/8.2 MB 3.2 MB/s eta 0:00:02
   ----------------------------- ---------- 6.0/8.2 MB 5.3 MB/s eta 0:00:01
   -----------


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Загрузка датасета

In [12]:
train = pd.read_csv('rusentitweet_train.csv')
test  = pd.read_csv('rusentitweet_test.csv')

print(train.shape)
print(train.columns.tolist())
print(train['label'].value_counts())
train.head()

(10713, 3)
['text', 'label', 'id']
neutral     4273
negative    2638
positive    1931
skip        1474
speech       397
Name: label, dtype: int64


,text,label,id
0,Помойму я вкрашилась в Чимина🤧 https://t.co/t2...,positive,1282311169534038016
1,@namaskaramsaroo Мотоцикль,neutral,1272864221202530309
2,Михаил Мишустин: меры по борьбе с коронавирусо...,neutral,1296860899739947008
3,@bbsbro_ доброй ноченьки 💕,speech,1288168112425246721
4,ну что пойду чекну фоточки,neutral,1287678712612364288


# Лемматизация 

In [13]:
import pymorphy2

morph = pymorphy2.MorphAnalyzer()

def preprocess_ru(text):
    tokens = str(text).lower().split()
    tokens = [morph.parse(t)[0].normal_form for t in tokens if t.isalpha()]
    return " ".join(tokens)

train['clean_text'] = train['text'].apply(preprocess_ru)
test['clean_text']  = test['text'].apply(preprocess_ru)

print("До:  ", train['text'][0])
print("После:", train['clean_text'][0])

До:   Помойму я вкрашилась в Чимина🤧 https://t.co/t2uZTS7NH2
После: помойма я вкрашиться в


# Загрузка русской модели 

In [14]:
ru_model = build_model(configs.classifiers.rusentiment_bert, download=True)
print("Модель загружена")

2026-04-27 19:09:35.117 INFO in 'deeppavlov.core.data.utils'['utils'] at line 97: Downloading from http://files.deeppavlov.ai/v1/classifiers/rusentiment_bert/rusentiment_bert_torch.tar.gz to C:\Users\Alex\.deeppavlov\models\classifiers\rusentiment_bert_torch.tar.gz
100%|██████████| 1.34G/1.34G [26:14<00:00, 853kB/s]  
2026-04-27 19:35:54.284 INFO in 'deeppavlov.core.data.utils'['utils'] at line 284: Extracting C:\Users\Alex\.deeppavlov\models\classifiers\rusentiment_bert_torch.tar.gz archive into C:\Users\Alex\.deeppavlov\models\classifiers\rusentiment_bert_torch
e:\labs\kafedra\.venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
e:\labs\kafedra\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default 

Модель загружена


# Проверка на примерах

In [15]:
samples = test.sample(10, random_state=42)

preds = ru_model(samples['clean_text'].tolist())

for text, true, pred in zip(samples['text'], samples['label'], preds):
    match = "✓" if pred == true else "✗"
    print(f"{match} [истина={true:>10}] [модель={pred}]")
    print(f"  {text[:80]}")
    print()

✓ [истина=   neutral] [модель=neutral]
  @E7TeHVjFCRAuFJw Может быть. Но интересно то, что сон повторяется часто и в тече

✗ [истина=      skip] [модель=neutral]
  @eddichka_mitro как это можно называть одинаково? это же разные штуки вообще htt

✗ [истина=  positive] [модель=neutral]
  Я испекла батон ))) https://t.co/nXOqRMdXSR

✗ [истина=  negative] [модель=neutral]
  @OnlinerBY А смысл? Через стор взять проблемно? Waste of time

✗ [истина=  negative] [модель=neutral]
  хелп меня сталкерят https://t.co/sxmWyBQmZX

✗ [истина=  positive] [модель=neutral]
  Стрим еще идет, буду рад

✓ [истина=   neutral] [модель=neutral]
  как МАГ может вернуть любимого без ГРЕХА?? ? и вените ли вы, что белая https://t

✓ [истина=  negative] [модель=negative]
  пиоресы суки не подали сигареты потому что только наичкой их можно опатитьц твар

✓ [истина=   neutral] [модель=neutral]
  Деловая Москва — О деловой жизни столицы https://t.co/wJcXuiLdhx

✗ [истина=   neutral] [модель=negative]
  @moonammin чонг